In [ ]:
!pip install numpy pandas pytesseract pillow matplotlib opencv-python openpyxl tensorflow scikit-learn seaborn gensim

In [ ]:
import numpy as np
import pandas as pd
import os
import re
from pathlib import Path

# Path dataset tetap sesuai Kaggle
dataset_dir = '/kaggle/input/datasets/nandaprasetia/datasetkomposisi114/dataset114'
image_extensions = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}

if not os.path.isdir(dataset_dir):
    raise FileNotFoundError(f'Dataset folder tidak ditemukan: {dataset_dir}')

print(f'Dataset folder: {dataset_dir}')


Dataset folder: ./dataset/


# image_paths

In [10]:
# Batch load image (tidak perlu satu per satu)
image_paths = sorted([
    str(p) for p in Path(dataset_dir).iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
])

print(f'Total image: {len(image_paths)}')
print('Contoh file:')
for p in image_paths[:5]:
    print('-', os.path.basename(p))


Total image: 114
Contoh file:
- Tamarin.jpeg
- astor_chocolate.jpg
- bakso ikan bernardi.jpg
- bakso sapi karawaci.jpg
- beng beng maxx.jpg


# import

In [11]:
import pytesseract
from PIL import Image
import matplotlib.pyplot as plt
import cv2


# output_folder

In [ ]:
output_folder = '/kaggle/working/ocr_output'
os.makedirs(output_folder, exist_ok=True)
print(f'Output folder: {output_folder}')


Output folder: ./ocr_output


# preprocess_image

In [13]:
def preprocess_image(path):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f'Gagal membaca gambar: {path}')

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.bilateralFilter(gray, 5, 75, 75)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Normalisasi foreground/background supaya OCR lebih stabil
    if np.mean(thresh) < 127:
        thresh = 255 - thresh

    kernel = np.ones((1, 1), np.uint8)
    processed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    return processed


def normalize_text(text):
    text = text.replace('\r', ' ')
    text = text.replace('\n', ' ')
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def clean_phrase(text):
    text = normalize_text(text.lower())
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def split_composition_items(composition_text):
    raw_items = re.split(r'[,;•|]', composition_text)
    items = []
    for item in raw_items:
        s = normalize_text(item)
        if len(s) >= 2:
            items.append(s)
    return items


def token_overlap_score(a, b):
    sa = set(clean_phrase(a).split())
    sb = set(clean_phrase(b).split())
    if not sa or not sb:
        return 0.0
    inter = len(sa.intersection(sb))
    return inter / max(1, min(len(sa), len(sb)))


def _ocr_lines(image_for_ocr, lang='eng', min_conf=35):
    data = pytesseract.image_to_data(image_for_ocr, lang=lang, output_type=pytesseract.Output.DICT)
    grouped = {}

    for i in range(len(data['text'])):
        word = (data['text'][i] or '').strip()
        if not word:
            continue
        try:
            conf = float(data['conf'][i])
        except Exception:
            continue
        if conf < min_conf:
            continue

        block = int(data['block_num'][i])
        par = int(data['par_num'][i])
        line = int(data['line_num'][i])
        left = int(data['left'][i])
        top = int(data['top'][i])
        width = int(data['width'][i])
        height = int(data['height'][i])

        key = (block, par, line)
        grouped.setdefault(key, []).append({
            'word': word,
            'left': left,
            'top': top,
            'right': left + width,
            'bottom': top + height,
            'height': height,
            'conf': conf,
            'block': block,
        })

    lines = []
    for key, words in grouped.items():
        words = sorted(words, key=lambda x: x['left'])
        text = normalize_text(' '.join(w['word'] for w in words))
        if not text:
            continue
        lines.append({
            'text': text,
            'norm': clean_phrase(text),
            'left': min(w['left'] for w in words),
            'right': max(w['right'] for w in words),
            'top': min(w['top'] for w in words),
            'bottom': max(w['bottom'] for w in words),
            'height': max(w['height'] for w in words),
            'conf': float(np.mean([w['conf'] for w in words])),
            'block': words[0]['block'],
        })

    lines.sort(key=lambda x: (x['top'], x['left']))
    return lines


def parse_composition_from_layout(path, processed_img):
    # Fokus ke area komposisi berdasarkan anchor keyword agar text luar tidak ikut
    lines = _ocr_lines(processed_img, lang='eng', min_conf=35)
    if not lines:
        return ''

    anchor_pattern = re.compile(r'\b(komposisi|composition|ingredients?)\b', re.I)
    stop_pattern = re.compile(
        r'\b(informasi nilai gizi|nutrition|takaran saji|energi total|cara penyimpanan|penyimpanan|'
        r'netto|berat bersih|expired|kedaluwarsa|bpom|kode produksi|saran penyajian|perhatian)\b',
        re.I,
    )

    anchors = [ln for ln in lines if anchor_pattern.search(ln['text'])]
    if not anchors:
        # fallback minimal: gunakan OCR full text yg dinormalisasi
        full_text = pytesseract.image_to_string(processed_img, lang='eng')
        return normalize_text(full_text)

    anchor = anchors[0]
    avg_h = np.mean([ln['height'] for ln in lines]) if lines else 20
    max_gap = max(18, int(avg_h * 1.8))

    selected = []
    started = False
    prev_bottom = None

    for ln in lines:
        if ln['top'] < anchor['top']:
            continue

        # mulai saat baris anchor ditemukan
        if not started and ln is anchor:
            started = True
            selected.append(ln['text'])
            prev_bottom = ln['bottom']
            continue

        if not started:
            continue

        # stop jika lompat vertikal terlalu jauh
        if prev_bottom is not None and (ln['top'] - prev_bottom) > max_gap:
            break

        # prefer area yang sebaris/sekolom dengan anchor
        horizontal_overlap = not (ln['right'] < anchor['left'] - 100 or ln['left'] > anchor['right'] + 900)
        near_anchor_column = abs(ln['left'] - anchor['left']) <= 220
        if not (horizontal_overlap or near_anchor_column):
            continue

        if stop_pattern.search(ln['text']):
            break

        selected.append(ln['text'])
        prev_bottom = ln['bottom']

    comp = normalize_text(' '.join(selected))
    comp = re.sub(r'(?i)^\s*(komposisi|composition|ingredients?)\s*[:\-]?\s*', '', comp).strip()
    return comp


def extract_bold_phrases(path, min_conf=45):
    img = cv2.imread(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    bin_inv = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 15
    )

    data = pytesseract.image_to_data(gray, lang='eng', output_type=pytesseract.Output.DICT)

    words = []
    for i in range(len(data['text'])):
        word = (data['text'][i] or '').strip()
        if not word:
            continue
        try:
            conf = float(data['conf'][i])
        except Exception:
            continue
        if conf < min_conf:
            continue

        l = int(data['left'][i]); t = int(data['top'][i])
        w = max(1, int(data['width'][i])); h = max(1, int(data['height'][i]))
        roi = bin_inv[t:t+h, l:l+w]
        ink_ratio = float(np.mean(roi > 0)) if roi.size else 0.0

        words.append({
            'word': word, 'ink': ink_ratio, 'h': h, 'left': l,
            'block': int(data['block_num'][i]), 'par': int(data['par_num'][i]), 'line': int(data['line_num'][i])
        })

    if not words:
        return []

    ink_thr = float(np.percentile([w['ink'] for w in words], 75))
    h_thr = float(np.percentile([w['h'] for w in words], 60))
    bold_words = [w for w in words if w['ink'] >= ink_thr and w['h'] >= h_thr]

    grouped = {}
    for w in bold_words:
        grouped.setdefault((w['block'], w['par'], w['line']), []).append(w)

    phrases = []
    for _, ws in grouped.items():
        ws = sorted(ws, key=lambda x: x['left'])
        phrase = normalize_text(' '.join(x['word'] for x in ws))
        if len(clean_phrase(phrase)) >= 2:
            phrases.append(phrase)

    seen = set(); out = []
    for p in phrases:
        k = clean_phrase(p)
        if k and k not in seen:
            seen.add(k); out.append(p)
    return out


def detect_bold_composition_items(path, composition_text):
    bold_phrases = extract_bold_phrases(path)
    composition_items = split_composition_items(composition_text)

    bold_items = []
    for item in composition_items:
        item_clean = clean_phrase(item)
        if not item_clean:
            continue
        for bp in bold_phrases:
            bp_clean = clean_phrase(bp)
            if not bp_clean:
                continue
            overlap = token_overlap_score(item, bp)
            contains = item_clean in bp_clean or bp_clean in item_clean
            if overlap >= 0.6 or contains:
                bold_items.append(item)
                break

    uniq = []
    seen = set()
    for x in bold_items:
        k = clean_phrase(x)
        if k and k not in seen:
            seen.add(k)
            uniq.append(x)

    label = 'unsafe' if uniq else 'safe'
    return label, uniq


In [14]:
records = []

for idx, path in enumerate(image_paths, start=1):
    base_name = os.path.splitext(os.path.basename(path))[0]
    print(f'[{idx}/{len(image_paths)}] Processing: {base_name}')

    processed = preprocess_image(path)

    # OCR full text (raw) tetap disimpan untuk audit
    raw_text = pytesseract.image_to_string(processed, lang='eng')

    # Komposisi diambil dari section/layout agar text luar tidak ikut
    composition_text = parse_composition_from_layout(path, processed)

    txt_filename = f'{base_name}.txt'
    output_path = os.path.join(output_folder, txt_filename)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(raw_text)

    label, bold_composition_items = detect_bold_composition_items(path, composition_text)

    records.append({
        'nama_produk': base_name,
        'text': composition_text,
        'label': label,
        'alergen_dari_teks_tebal': '; '.join(bold_composition_items),
    })

df_dataset = pd.DataFrame(records)
print('\nSelesai build dataset:')
print(df_dataset.head(10))


[1/114] Processing: Tamarin
[2/114] Processing: astor_chocolate
[3/114] Processing: bakso ikan bernardi
[4/114] Processing: bakso sapi karawaci
[5/114] Processing: beng beng maxx
[6/114] Processing: bihunku rasa ayam bawang
[7/114] Processing: bihunku rasa soto
[8/114] Processing: biokul greek yoghurt
[9/114] Processing: biokul yoghurt ori
[10/114] Processing: blastoz bites
[11/114] Processing: boncabe_makaroni_krispi_level_15
[12/114] Processing: champ chicken stick
[13/114] Processing: cheese cream meiji
[14/114] Processing: chitato
[15/114] Processing: choco chips
[16/114] Processing: choco mania
[17/114] Processing: choco pie
[18/114] Processing: cizzring
[19/114] Processing: corntoz_jagung_bakar
[20/114] Processing: croissant creamy chocolate
[21/114] Processing: deka crepes
[22/114] Processing: delfiorion_custas
[23/114] Processing: enaak
[24/114] Processing: fiesta spicy karage
[25/114] Processing: fiesta spicy wing
[26/114] Processing: fitbar
[27/114] Processing: folabee
[28/11

In [15]:
# Tampilkan dataframe final (nama produk, text/komposisi, label)
df_final = df_dataset[['nama_produk', 'text', 'label']].copy()
display(df_final.head(20))
print(f'Total baris: {len(df_final)}')


,nama_produk,text,label
0,Tamarin,"Guia, Glukosa, Pengatur Keasaman (Asam Sitrat)...",safe
1,astor_chocolate,"@® Komposisi: Gula, Tepung Terigu, Minyak Naba...",unsafe
2,bakso ikan bernardi,epeeeeeet tT TT yee rasemeisi e BAGING IMAM (8...,safe
3,bakso sapi karawaci,"Daging Sapi (70%), Es Batu, Tepung Tapioka (Me...",unsafe
4,beng beng maxx,"Glukosa, Lemak Nabati mengandung Antioksidan B...",unsafe
5,bihunku rasa ayam bawang,"KAMPOSISi : Bihun : Pati Jagung, Beras, Pati T...",unsafe
6,bihunku rasa soto,"Bihun : Pati Jagung, Beras, Pati Tapioka, Peng...",unsafe
7,biokul greek yoghurt,"Av, Konsentret Susu, Bubuk, (sim Syeu, Pett Pe...",unsafe
8,biokul yoghurt ori,"Air, Susu Sapi (25%), Susu Skim Bubuk. Konsent...",safe
9,blastoz bites,"Gula, Pengganti Lemak Kakao, Tepung Mer (Menga...",unsafe


Total baris: 114


In [16]:
# Export dataset ke CSV dan XLSX
csv_path = os.path.join(output_folder, 'dataset_komposisi_label.csv')
xlsx_path = os.path.join(output_folder, 'dataset_komposisi_label.xlsx')

df_final.to_csv(csv_path, index=False, encoding='utf-8')
df_final.to_excel(xlsx_path, index=False)

print('CSV :', csv_path)
print('XLSX:', xlsx_path)


ModuleNotFoundError: No module named 'openpyxl'

## Pemodelan NLP (Word2Vec + BiLSTM TensorFlow)
Section ini menggunakan `df_final` hasil OCR untuk klasifikasi `safe` vs `unsafe`.


In [ ]:
import random
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

from gensim.models import Word2Vec

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)


In [ ]:
# Siapkan dataset untuk pemodelan
df_model = df_final.copy()
df_model['text'] = df_model['text'].fillna('').astype(str)
df_model = df_model[df_model['text'].str.strip() != ''].copy()

label_encoder = LabelEncoder()
df_model['label_id'] = label_encoder.fit_transform(df_model['label'])

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df_model['text'].tolist(),
    df_model['label_id'].values,
    test_size=0.2,
    random_state=SEED,
    stratify=df_model['label_id'].values
)

print('Train size:', len(X_train_text))
print('Test size :', len(X_test_text))
print('Classes   :', list(label_encoder.classes_))


In [ ]:
# Tokenisasi + Word2Vec
def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return [tok for tok in text.split() if tok]

train_tokens = [simple_tokenize(t) for t in X_train_text]
test_tokens = [simple_tokenize(t) for t in X_test_text]
all_texts = X_train_text + X_test_text

VOCAB_SIZE = 20000
MAX_LEN = 120
EMBED_DIM = 100

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(all_texts)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=EMBED_DIM,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=20,
    seed=SEED
)

num_words = min(VOCAB_SIZE, len(tokenizer.word_index) + 1)
embedding_matrix = np.random.normal(scale=0.6, size=(num_words, EMBED_DIM)).astype(np.float32)
embedding_matrix[0] = np.zeros((EMBED_DIM,), dtype=np.float32)

for word, idx in tokenizer.word_index.items():
    if idx >= num_words:
        continue
    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]

print('Vocabulary size (tokenizer):', len(tokenizer.word_index))
print('Num words used             :', num_words)
print('Embedding matrix shape     :', embedding_matrix.shape)


In [ ]:
# BiLSTM model (TensorFlow)
model = Sequential([
    Embedding(
        input_dim=num_words,
        output_dim=EMBED_DIM,
        weights=[embedding_matrix],
        input_length=MAX_LEN,
        trainable=True
    ),
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
] )

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

history = model.fit(
    X_train_pad,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

model.summary()


In [ ]:
# Generate training logs: accuracy/f1 dan loss
hist = pd.DataFrame(history.history)

if {'precision', 'recall'}.issubset(hist.columns):
    hist['f1'] = 2 * (hist['precision'] * hist['recall']) / (hist['precision'] + hist['recall'] + 1e-8)
if {'val_precision', 'val_recall'}.issubset(hist.columns):
    hist['val_f1'] = 2 * (hist['val_precision'] * hist['val_recall']) / (hist['val_precision'] + hist['val_recall'] + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(hist['accuracy'], label='train_accuracy')
if 'val_accuracy' in hist.columns:
    axes[0].plot(hist['val_accuracy'], label='val_accuracy')
if 'f1' in hist.columns:
    axes[0].plot(hist['f1'], label='train_f1')
if 'val_f1' in hist.columns:
    axes[0].plot(hist['val_f1'], label='val_f1')
axes[0].set_title('Training Log Accuracy/F1')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Score')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(hist['loss'], label='train_loss')
if 'val_loss' in hist.columns:
    axes[1].plot(hist['val_loss'], label='val_loss')
axes[1].set_title('Training Log Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

history_csv_path = os.path.join(output_folder, 'training_history_bilstm_word2vec.csv')
hist.to_csv(history_csv_path, index=False)
print('Training history saved to:', history_csv_path)


In [ ]:
# Evaluasi model pada test set
y_prob = model.predict(X_test_pad).ravel()
y_pred = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)

eval_table = pd.DataFrame([
    {'metric': 'accuracy', 'value': acc},
    {'metric': 'precision', 'value': prec},
    {'metric': 'recall', 'value': rec},
    {'metric': 'f1_score', 'value': f1},
])

print('Evaluation Table:')
display(eval_table)

target_names = list(label_encoder.classes_)
report_dict = classification_report(y_test, y_pred, target_names=target_names, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose().reset_index().rename(columns={'index': 'label'})

print('Classification Report Table:')
display(report_df)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

pred_table = pd.DataFrame({
    'text': X_test_text,
    'label_actual': label_encoder.inverse_transform(y_test),
    'label_pred': label_encoder.inverse_transform(y_pred),
    'score_unsafe': y_prob
})

eval_csv = os.path.join(output_folder, 'evaluation_table_bilstm_word2vec.csv')
eval_xlsx = os.path.join(output_folder, 'evaluation_table_bilstm_word2vec.xlsx')
report_csv = os.path.join(output_folder, 'classification_report_bilstm_word2vec.csv')
pred_csv = os.path.join(output_folder, 'predictions_test_bilstm_word2vec.csv')

eval_table.to_csv(eval_csv, index=False)
eval_table.to_excel(eval_xlsx, index=False)
report_df.to_csv(report_csv, index=False)
pred_table.to_csv(pred_csv, index=False)

print('Saved:', eval_csv)
print('Saved:', eval_xlsx)
print('Saved:', report_csv)
print('Saved:', pred_csv)
